In [1]:
!git clone https://github.com/liunian-Jay/GainRAG.git
!pip install -q -U transformers accelerate bitsandbytes datasets

import sys
sys.path.insert(0, "/content/GainRAG/GainRAG/gainRAG")  # so the repo's own imports resolve

Cloning into 'GainRAG'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 73 (delta 10), reused 16 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 228.74 KiB | 1.22 MiB/s, done.
Resolving deltas: 100% (10/10), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.2 MB/s eta 0:00:00


In [2]:
from llm_supervision.decoding import ContrastiveTool           # Eq. 4/5, verbatim from the paper's authors
from rag_workflow.prompts import get_input_with_R_Kth, INSTRUCTION_PROMPT, TASK_INST
from llm_inference.build_prompts import llm_prompts             # applies the Llama-3 chat template

contrastive_tool = ContrastiveTool()

In [3]:
from datasets import load_dataset
import json
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation[:5]")
rows = []
for ex in ds:
    sup_titles = set(ex["supporting_facts"]["title"])
    ctxs = []
    for title, sents in zip(ex["context"]["title"], ex["context"]["sentences"]):
        text = " ".join(sents)
        ctxs.append({
            "title": title,
            "text": text,
            "hasanswer": ex["answer"].lower() in text.lower(),
            "score": 0.0,
        })
    rows.append({"question": ex["question"], "answers": [ex["answer"]], "ctxs": ctxs})
with open("hotpotqa_mini.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")
print(f"{len(rows)} questions, {len(rows[0]['ctxs'])} candidate passages for Q1")



README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

5 questions, 10 candidate passages for Q1


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "NousResearch/Meta-Llama-3-8B-Instruct"  # ungated mirror of the same weights

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model.eval()

lm_type = "Llama-3-8B-Instruct"  # tells llm_prompts() which chat template to apply

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [5]:
prompts_data = get_input_with_R_Kth("hotpotqa_mini.jsonl", k=10, task="HotpotQA")

results_per_question = []
for item in prompts_data:
    standard_prompt = item["prompt_standard"]
    retrieved_prompts = [p["prompt_retrieved"] for p in item["passages"]]

    standard_query = llm_prompts(lm_type, standard_prompt, tokenizer, tokenize=False)[0]
    retrieved_queries = llm_prompts(lm_type, retrieved_prompts, tokenizer, tokenize=False)

    gold_answer = item["answers"][0]
    # exact call construct_hf.py makes — batched contrastive-decoded PPL over all passages
    gain_list = contrastive_tool.contrastive_PPL_multi_pro(
        model, tokenizer, standard_query, retrieved_queries, gold_answer, alpha=0.5
    )
    for p, gain in zip(item["passages"], gain_list):
        p["PPL_CD"] = gain
    results_per_question.append(item)

print("Done —", len(results_per_question), "questions scored")

Done — 5 questions scored


In [6]:
import pandas as pd

q0 = results_per_question[0]
print("Q:", q0["question"])
print("Gold answer:", q0["answers"][0])

df = pd.DataFrame([
    {"title": p["title"], "has_answer_substring": p["has_answer"], "gain_PPL_CD": round(p["PPL_CD"], 3)}
    for p in q0["passages"]
]).sort_values("gain_PPL_CD")

df

Q: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes


,title,has_answer_substring,gain_PPL_CD
8,Conrad Brooks,False,3.704282e+06
1,Scott Derrickson,False,1.465072e+07
6,Adam Collis,False,2.737115e+07
4,Ed Wood,False,9.553469e+07
2,"Woodson, Arkansas",False,1.390022e+08
0,Ed Wood (film),False,2.291758e+08
3,Tyler Bates,False,1.918866e+09
9,Doctor Strange (2016 film),False,2.062975e+10
7,Sinister (film),False,5.607747e+10
5,Deliver Us from Evil (2014 film),False,9.245612e+10
